# 05 · Outputs

What a computation returns and what can be derived from it without recomputing:

1. the result dict of `compute_cumulants`: means ($k = 1$), $C(\tau)$ ($k = 2$), loop orders,
2. the `Cumulant` object: spectrum, integrals against kernels, mode decomposition,
3. three-point functions ($k = 3$),
4. symbolic results: loop corrections as exact rational functions of the parameters,
5. switch-on transients: statistics at finite time after the noise is turned on.

Model: an asymmetric anharmonic oscillator, so that the mean and the third cumulant are nonzero,

$$\dot x = -\mu x - a x^2 - \varepsilon x^3 + \xi,\qquad \langle\xi\xi\rangle = 2D\delta.$$

Run time: about three minutes.

In [ ]:
%matplotlib inline
try:
    preparser(False)   # Sage kernel: plain Python syntax and numbers in the cells below
except NameError:
    pass
import os, sys, time, warnings
import numpy as np
import matplotlib.pyplot as plt
# locate the repository root (the folder holding the 'api' package) from any working directory
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'api')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(_root)
def ells(r):   # C(tau) per loop order (real parts), a list indexed by the loop order
    return [np.real(np.asarray(r['C_tau_by_ell'][l])) for l in sorted(r['C_tau_by_ell'])]
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
from api.sde import SDE
from api.noise import Gaussian
from api import compute_cumulants
import daedalus as dd

m = (SDE('asymmetric oscillator')
     .parameter('mu', default=1.0, domain='positive').parameter('a', default=0.2)
     .parameter('eps', default=0.05, domain='positive').parameter('D', default=0.5, domain='positive')
     .field('x')
     .drift('x', '-mu*x - a*x^2 - eps*x^3')
     .noise('xi', Gaussian(var='2*D'), couples={'x': 1}))
model = m.build()
X = ('x', 1)

## 1. The result dict

Keys used most: `C_tau` (total, on `tau_grid`), `C_tau_by_ell` (per loop order), `mf_values` (the mean-field saddle), `status`, `analysis` (the model report). For $k = 1$ the loop corrections shift the mean away from the saddle.

In [ ]:
r1 = compute_cumulants(model, k=1, max_ell=2, external_fields=[X], tau_grid=np.array([0.0]),
                       parallel=False, verbose=False)
xs = float(np.atleast_1d(r1['mf_values']['xstar'])[0])
print('saddle x* =', xs)
for ell in sorted(r1['C_tau_by_ell']):
    print(f"  loop order {ell}: mean correction {complex(r1['C_tau_by_ell'][ell][0]).real:+.5f}")
tau = np.linspace(-6, 6, 49)
r2 = compute_cumulants(model, k=2, max_ell=1, external_fields=[X, X], tau_grid=tau,
                       parallel=False, verbose=False)
print('status', r2['status'], '| keys:', sorted(k for k in r2 if not k.startswith('_'))[:20], '...')

## 2. The `Cumulant` object

`dd.cumulant(result)` wraps a result without recomputing. For a $k = 2$ result on the state-space route the smooth part is held as matrix-exponential triples, so it can be evaluated at any lag, Fourier-transformed and integrated against kernels exactly:

* `c.me(tau)`: $C(\tau)$ at arbitrary lags,
* `c.spectrum(omega)`: $S(\omega) = \int e^{-i\omega\tau} C(\tau)\,\mathrm d\tau$,
* `c.integrate_against(kernel)`: $\int L(\tau)\,C(\tau)\,\mathrm d\tau$ for exponential (times polynomial) kernels $L$,
* `c.modes`: $C(\tau)$ as a sum of terms $c\,\tau^p e^{r\tau}$ per loop order.

In [ ]:
from api.symbolic_out import Kernel as LKernel
c = dd.cumulant(r2)
fine = np.linspace(-6, 6, 401)
omega = np.linspace(0, 6, 200)
S = np.real(c.spectrum(omega))
L = LKernel.exponential(1.0, -0.5)               # L(s) = exp(-s/2) for s > 0
I = c.integrate_against(L)
print('∫ e^{-s/2} C(s) ds over s > 0:', I)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(fine, np.real(c.me(fine)), label='c.me(τ) (any lag)')
ax[0].plot(tau, np.real(r2['C_tau']), 'o', ms=3, label='C_tau (grid)')
ax[0].set_xlabel(r'$\tau$'); ax[0].legend()
ax[1].plot(omega, S); ax[1].set_xlabel(r'$\omega$'); ax[1].set_ylabel(r'$S(\omega)$')
plt.tight_layout(); plt.show()
for ell, terms in c.modes.items():
    print(f"loop order {ell}: {len(terms)} mode terms, e.g. {terms[0]}")

## 3. Three-point functions

For $k = 3$ the result holds callables of the external times: `total_C_by_ell[ell](t1, t2, t3)`. Below, the slice $C_3(0, 0, \tau) = \kappa(x(0), x(0), x(\tau))$.

In [ ]:
t0 = time.time()
r3 = compute_cumulants(model, k=3, max_ell=1, external_fields=[X, X, X], tau_grid=np.linspace(-4, 4, 9),
                       parallel=False, verbose=False)
print(f"{time.time() - t0:.0f} s")
T = np.linspace(-4, 4, 41)
C3 = {ell: np.array([complex(r3['total_C_by_ell'][ell](0.0, 0.0, t)).real for t in T]) for ell in (0, 1)}
plt.figure(figsize=(6, 3.4))
plt.plot(T, C3[0], 'k--', label='tree'); plt.plot(T, C3[0] + C3[1], label='tree + 1 loop')
plt.xlabel(r'$\tau$'); plt.ylabel(r'$C_3(0,0,\tau)$'); plt.legend(); plt.show()

## 4. Symbolic results

`dd.symbolic_cumulant` returns the equal-time cumulant of each loop order as an exact rational function of the parameters you mark symbolic (any $k$; the pipeline is run once at the numeric reference point and the symbolic result is checked against it). For the symmetric quartic oscillator ($a = 0$) the one- and two-loop variance corrections are the exact Boltzmann coefficients $-3\varepsilon D^2/\mu^3$ and $24\varepsilon^2 D^3/\mu^5$.

In [ ]:
sym = (SDE('quartic oscillator')
       .parameter('mu', default=1.0, domain='positive').parameter('eps', default=0.05, domain='positive')
       .parameter('D', default=0.5, domain='positive')
       .field('x').drift('x', '-mu*x - eps*x^3')
       .noise('xi', Gaussian(var='2*D'), couples={'x': 1}))
sc = dd.symbolic_cumulant(sym.build(), k=2, max_ell=2, external_fields=[X, X], symbols=['mu', 'eps', 'D'])
for ell in sorted(sc.by_ell):
    print(f"loop order {ell}:", sc.expression(ell))

The same for the third cumulant of the asymmetric model at one loop, with $a$ and $\varepsilon$ symbolic (the other parameters enter as numbers):

In [ ]:
s3 = dd.symbolic_cumulant(model, k=3, max_ell=1, external_fields=[X, X, X], symbols=['a', 'eps'])
for ell in sorted(s3.by_ell):
    print(f"loop order {ell}:", s3.expression(ell))
print('notes:', s3.notes)

Exports: `.latex(ell)`, `.sympy(ell)`, and `dd.symbolic_triple` for the matrix-exponential form of $C(\tau)$.

## 5. Switch-on transients

With `t_min=` and `times=` the fluctuations start at zero at the absolute time $t_{\min}$ (the system sits at its mean-field state and the noise is switched on). The cumulant is then a function of the absolute external times, returned in `C_times`. The equal-time variance grows to its stationary value:

In [ ]:
times = np.linspace(0.0, 4.0, 17)
rt = compute_cumulants(sym.build(), k=2, max_ell=1, external_fields=[X, X], t_min=0.0,
                       times=np.array([[t, t] for t in times]), parallel=False, verbose=False)
rs = compute_cumulants(sym.build(), k=2, max_ell=1, external_fields=[X, X], tau_grid=np.array([0.0]),
                       parallel=False, verbose=False)
plt.figure(figsize=(6, 3.4))
plt.plot(times, np.real(rt['C_times']), 'o-', ms=3, label=r'$\mathrm{Var}\,x(t)$, switched on at $t=0$')
plt.axhline(np.real(rs['C_tau'][0]), color='k', ls='--', label='stationary')
plt.xlabel('t'); plt.legend(); plt.show()